# Task 2: First CDC events to Bronze

Read CDC events from Kafka in Spark, parse the Debezium envelope, and write them append-only to a Bronze Iceberg table.

## Step 1: Configure Spark session with Iceberg REST catalog

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("CDC-Bronze") \
  .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.lakehouse.type", "rest") \
  .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181") \
  .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
  .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000") \
  .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true") \
  .config("spark.sql.defaultCatalog", "lakehouse") \
  .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")

## Step 2: Read from Kafka CDC topic

In [ ]:
raw = spark.read \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "kafka:9092") \
  .option("subscribe", "dbserver1.public.customers") \
  .option("startingOffsets", "earliest") \
  .load()

print(f"Raw records count: {raw.count()}")
raw.printSchema()

## Step 3: Parse Debezium envelope and extract CDC fields

In [ ]:
from pyspark.sql import functions as F

raw_filtered = raw.filter(F.col("value").isNotNull())

bronze_df = raw_filtered.select(
  F.col("topic"),
  F.col("partition").alias("kafka_partition"),
  F.col("offset").alias("kafka_offset"),
  F.col("timestamp").alias("kafka_timestamp"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("long").alias("ts_ms"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.name").alias("after_name"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.email").alias("after_email"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.country").alias("after_country"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
)

print("Bronze DataFrame schema:")
bronze_df.printSchema()

## Step 4: Write to Bronze Iceberg table (append-only)

In [ ]:
bronze_df.writeTo("lakehouse.cdc.bronze_customers").createOrReplace()
print("Bronze table created successfully")

## Step 5: Verify Bronze table contents

In [ ]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.bronze_customers").show()
spark.sql("""
  SELECT op, after_id, after_name, after_email, ts_ms
  FROM lakehouse.cdc.bronze_customers LIMIT 5
""").show(truncate=False)